In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import StandardScaler

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D2 in response_OUS
data = list(OUS_D2['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D2, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [5]:
# Check null values in D2 
clinical_train.isnull().sum().sum()

0

In [6]:
""" Takes too long time 
# Check collinearity for OUS data 
df = clinical_train

# Create a correlation matrix
correlation_matrix = df.corr()

plt.figure(figsize=(15, 12))

# Plot a heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.show()
""" 

" Takes too long time \n# Check collinearity for OUS data \ndf = clinical_train\n\n# Create a correlation matrix\ncorrelation_matrix = df.corr()\n\nplt.figure(figsize=(15, 12))\n\n# Plot a heatmap\nsns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)\nplt.show()\n"

## Test dataset: MAASTRO 

In [7]:
(MAASTRO_D2['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [8]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [9]:
# need to choose patient_id from MAASTRO_D2 in response_MAASTRO
data = list(MAASTRO_D2['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [10]:
# Merge MAASTRO_D2 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D2, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'DFS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,shape_Elongation,shape_Flatness,shape_LeastAxisLength,shape_MajorAxisLength,shape_Maximum2DDiameterColumn,shape_Maximum2DDiameterRow,shape_Maximum2DDiameterSlice,shape_Maximum3DDiameter,shape_MeshVolume,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,OS,OS_event
0,1,0.765178,0.610062,30.504395,50.002093,52.886671,58.000000,47.759816,58.864251,38067.208333,...,0.028808,0.837387,0.000026,0.001705,0.122209,0.000026,0.008291,0.000341,62.43,0.0
1,2,0.776540,0.504616,21.069386,41.753334,44.922155,44.294469,44.147480,48.723711,17870.791667,...,0.049615,0.806842,0.000167,0.002958,0.128976,0.000167,0.008148,0.000335,60.00,0.0
2,3,0.697164,0.478604,21.238275,44.375483,48.466483,45.891176,37.656341,48.969378,17426.500000,...,0.019514,0.830272,0.000057,0.001831,0.137282,0.000000,0.008641,0.000229,44.43,1.0
3,4,0.574636,0.446059,20.570459,46.115989,42.544095,47.507894,33.837849,55.226805,12463.791667,...,0.047155,0.760949,0.000000,0.003597,0.171595,0.000080,0.013187,0.000240,37.20,1.0
4,6,0.633419,0.480378,26.130155,54.394967,66.030296,56.320511,45.398238,67.089492,28300.125000,...,0.033990,0.824865,0.000000,0.002116,0.128134,0.000035,0.009379,0.000529,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,0.882411,0.577884,19.774388,34.218615,41.785165,40.261644,35.805028,43.520110,12822.458333,...,0.017489,0.834590,0.000000,0.001321,0.137194,0.000078,0.008706,0.000078,19.00,1.0
95,111,0.535802,0.455642,23.259099,51.046869,51.264022,51.264022,33.600595,52.440442,18368.291667,...,0.029979,0.821431,0.000000,0.000543,0.137620,0.000054,0.008907,0.000489,85.87,1.0
96,112,0.716610,0.631485,31.838169,50.417953,46.324939,56.035703,57.070132,57.671483,35384.541667,...,0.017354,0.859957,0.000000,0.000988,0.115099,0.000028,0.005700,0.000141,42.87,0.0
97,113,0.665145,0.628338,28.213278,44.901412,50.289164,50.596443,35.777088,51.478151,26180.208333,...,0.025399,0.847908,0.000000,0.001487,0.117654,0.000000,0.005759,0.000038,58.93,0.0


In [11]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [12]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,shape_Elongation,shape_Flatness,shape_LeastAxisLength,shape_MajorAxisLength,shape_Maximum2DDiameterColumn,shape_Maximum2DDiameterRow,shape_Maximum2DDiameterSlice,shape_Maximum3DDiameter,shape_MeshVolume,shape_MinorAxisLength,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,OS,OS_event


In [13]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS'])]

# y 
y = clinical_train.loc[:, ['OS', 'event_OS']]

# Set lower, upper time point and times for IBS calculation later 
lower, upper = np.percentile(y['OS'], [10, 90])
times = np.arange(lower, upper)

In [14]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [15]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 374)
y_train:  (139,)


In [16]:
clinical_test.rename(columns = {'OS_event' : 'event_OS'}, inplace = True)

In [17]:
# X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'event_OS'])]

# y y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['OS', 'event_OS']]
lower, upper = np.percentile(y_MAASTRO['OS'], [10, 90])
times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_OS'], y_MAASTRO['OS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 376)

# Feature Selection

In [18]:
# Choose features from the result of Cox PLSR in R
selected_features = [
"shape_MajorAxisLength",                            
"LBP_120_PET",                                     
"LBP_201_PET",                   
"glszm_SmallAreaLowGrayLevelEmphasis_CT_c16",
"shape_Maximum3DDiameter",
"shape_SurfaceVolumeRatio",                  
"shape_Sphericity",               
"shape_Elongation",                           
"gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2"    
]

In [19]:
X_plsr = X.loc[:, selected_features]
X_new = X_plsr.copy()

In [20]:
# Selecct the columns from X_MAASTRO
MAASTRO_new = X_MAASTRO.loc[:, selected_features]

# Standardization

In [21]:
# Copy the original X for later 
original_X = X.copy()

In [22]:
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Standardize X_new, the new data with the selected features only 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
scaler = StandardScaler() 
X_new_numeric_columns = X_new_numeric.columns
X_new_numeric_index = X_new_numeric.index 
X_new_numeric_std = scaler.fit_transform(X_new_numeric)
X_new_numeric_std = pd.DataFrame(X_new_numeric_std,
                                 columns=X_new_numeric_columns, 
                                 index=X_new_numeric_index)
X_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]

In [23]:
# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns

MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [24]:
X_new

,shape_MajorAxisLength,LBP_120_PET,LBP_201_PET,glszm_SmallAreaLowGrayLevelEmphasis_CT_c16,shape_Maximum3DDiameter,shape_SurfaceVolumeRatio,shape_Sphericity,shape_Elongation,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2
0,42.073251,0.140311,0.000062,0.029425,47.339202,0.251218,0.761164,0.600926,0.000959
1,24.613845,0.191058,0.000349,0.037915,28.106939,0.489853,0.697049,0.841579,0.002776
2,48.030294,0.126531,0.000000,0.008009,60.049979,0.278467,0.565792,0.772821,0.001179
3,25.589900,0.192388,0.000000,0.018398,32.572995,0.474018,0.684364,0.847727,0.002748
4,34.684750,0.202073,0.000399,0.013051,39.962482,0.563135,0.503142,0.831483,0.002309
...,...,...,...,...,...,...,...,...,...
134,33.069705,0.152626,0.000000,0.014938,37.696154,0.322021,0.742102,0.680294,0.001347
135,41.043692,0.142778,0.000079,0.011441,52.430907,0.227705,0.722918,0.758193,0.000850
136,36.618802,0.140582,0.000000,0.020431,42.743421,0.298398,0.652963,0.770113,0.001111
137,45.870392,0.156640,0.000054,0.019663,51.536395,0.252893,0.724255,0.628897,0.000935


In [25]:
X_new_std

,shape_MajorAxisLength,LBP_120_PET,LBP_201_PET,glszm_SmallAreaLowGrayLevelEmphasis_CT_c16,shape_Maximum3DDiameter,shape_SurfaceVolumeRatio,shape_Sphericity,shape_Elongation,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2
0,-0.098422,-0.476493,-0.315438,1.274405,-0.205326,-0.558082,1.075042,-0.681983,-0.496657
1,-1.209119,1.084993,1.967842,2.273964,-1.196220,1.585077,0.242767,1.006304,2.054251
2,0.280542,-0.900502,-0.805067,-1.246890,0.449566,-0.313359,-1.461057,0.523938,-0.187799
3,-1.147026,1.125919,-0.805067,-0.023759,-0.966118,1.442865,0.078112,1.049432,2.014276
4,-0.568449,1.423898,2.359078,-0.653294,-0.585393,2.243217,-2.274305,0.935477,1.398595
...,...,...,...,...,...,...,...,...,...
134,-0.671191,-0.097562,-0.805067,-0.431120,-0.702160,0.077795,0.827589,-0.125182,0.048554
135,-0.163918,-0.400566,-0.180035,-0.842806,0.057012,-0.769256,0.578577,0.421312,-0.649927
136,-0.445412,-0.468144,-0.805067,0.215575,-0.442112,-0.134362,-0.329498,0.504939,-0.282932
137,0.143137,0.025938,-0.374777,0.125178,0.010924,-0.543043,0.595927,-0.485754,-0.529934


In [26]:
MAASTRO_new 

,shape_MajorAxisLength,LBP_120_PET,LBP_201_PET,glszm_SmallAreaLowGrayLevelEmphasis_CT_c16,shape_Maximum3DDiameter,shape_SurfaceVolumeRatio,shape_Sphericity,shape_Elongation,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2
0,50.002093,0.122209,0.000026,0.010495,58.864251,0.215184,0.668072,0.765178,0.000768
1,41.753334,0.128976,0.000167,0.035018,48.723711,0.276092,0.669961,0.776540,0.001213
2,44.375483,0.137282,0.000000,0.012819,48.969378,0.298887,0.624081,0.697164,0.001176
3,46.115989,0.171595,0.000080,0.029974,55.226805,0.361096,0.577624,0.574636,0.001192
4,54.394967,0.128134,0.000035,0.013458,67.089492,0.251519,0.630933,0.633419,0.000766
...,...,...,...,...,...,...,...,...,...
94,34.218615,0.137194,0.000078,0.039092,43.520110,0.307574,0.671754,0.882411,0.001137
95,51.046869,0.137620,0.000054,0.015831,52.440442,0.289922,0.632189,0.535802,0.000962
96,50.417953,0.115099,0.000028,0.011031,57.671483,0.228184,0.645548,0.716610,0.000621
97,44.901412,0.117654,0.000000,0.019801,51.478151,0.223872,0.727488,0.665145,0.000947


In [27]:
MAASTRO_new_std

,shape_MajorAxisLength,LBP_120_PET,LBP_201_PET,glszm_SmallAreaLowGrayLevelEmphasis_CT_c16,shape_Maximum3DDiameter,shape_SurfaceVolumeRatio,shape_Sphericity,shape_Elongation,gldm_SmallDependenceLowGrayLevelEmphasis_d_1_PET_b2
0,0.405980,-1.033478,-0.596780,-0.954258,0.388474,-0.881698,-0.133381,0.470316,-0.764236
1,-0.118774,-0.825249,0.524128,1.932841,-0.133992,-0.334696,-0.108854,0.550025,-0.139994
2,0.048037,-0.569695,-0.805067,-0.680666,-0.121335,-0.129977,-0.704414,-0.006831,-0.191744
3,0.158761,0.486120,-0.170569,1.338998,0.201064,0.428717,-1.307469,-0.866418,-0.168865
4,0.685437,-0.851180,-0.525146,-0.605333,0.812259,-0.555379,-0.615466,-0.454030,-0.767510
...,...,...,...,...,...,...,...,...,...
94,-0.598102,-0.572399,-0.187979,2.412550,-0.402095,-0.051953,-0.085581,1.292755,-0.246354
95,0.472444,-0.559284,-0.373913,-0.326069,0.057503,-0.210490,-0.599161,-1.138850,-0.492861
96,0.432435,-1.252248,-0.581052,-0.891107,0.327020,-0.764952,-0.425762,0.129595,-0.971544
97,0.081495,-1.173644,-0.805067,0.141405,0.007923,-0.803671,0.637891,-0.231458,-0.513058


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [28]:
# Setting the y format for skf below  
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 01:13:11,330] A new study created in memory with name: no-name-5af1276c-e721-4314-99b2-8f10d6764512


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6666666666666666
Fold 2 C-index: 0.7053571428571429


[I 2024-04-16 01:13:28,393] A new study created in memory with name: no-name-93b9b893-8f15-42ca-9f1c-9ba26aa4a6f3


Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.7637130801687764
Fold 5 C-index: 0.7323943661971831
[I 2024-04-16 01:13:28,382] Trial 0 finished with value: 0.7353909570603067 and parameters: {}. Best is trial 0 with value: 0.7353909570603067.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7353909570603067], datetime_start=datetime.datetime(2024, 4, 16, 1, 13, 11, 497359), datetime_complete=datetime.datetime(2024, 4, 16, 1, 13, 28, 382078), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7353909570603067


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.19333371135654168
Fold 2 IBS: 0.15067618719150463
Fold 3 IBS: 0.16491771324415705
Fold 4 IBS: 0.14041209978706223
Fold 5 IBS: 0.15824636706666786
[I 2024-04-16 01:13:29,022] Trial 0 finished with value: 0.1615172157291867 and parameters: {}. Best is trial 0 with value: 0.1615172157291867.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.1615172157291867], datetime_start=datetime.datetime(2024, 4, 16, 1, 13, 28, 502183), datetime_complete=datetime.datetime(2024, 4, 16, 1, 13, 29, 21943), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.1615172157291867


In [29]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [30]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.735
train_ibs:  0.162


#### Test

In [31]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [32]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.558
IBS score: 0.252


In [33]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [34]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis

#### Train

In [35]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 01:13:29,422] A new study created in memory with name: no-name-e5c69fcf-9dea-4f85-aba4-27ae92a1f337


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.645021645021645
Fold 2 C-index: 0.7477678571428571
Fold 3 C-index: 0.7524509803921569
Fold 4 C-index: 0.7236286919831224


[I 2024-04-16 01:13:29,814] A new study created in memory with name: no-name-9737969a-741f-4fa6-8cb7-682c10a4d9b6


Fold 5 C-index: 0.7206572769953051
[I 2024-04-16 01:13:29,807] Trial 0 finished with value: 0.7179052903070173 and parameters: {}. Best is trial 0 with value: 0.7179052903070173.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7179052903070173], datetime_start=datetime.datetime(2024, 4, 16, 1, 13, 29, 499101), datetime_complete=datetime.datetime(2024, 4, 16, 1, 13, 29, 807236), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7179052903070173


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.2139765126020755
Fold 2 IBS: 0.22157789890073135
Fold 3 IBS: 0.20453593468821316
Fold 4 IBS: 0.22473802683456762
Fold 5 IBS: 0.2181243002990223
[I 2024-04-16 01:13:30,197] Trial 0 finished with value: 0.216590534664922 and parameters: {}. Best is trial 0 with value: 0.216590534664922.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.216590534664922], datetime_start=datetime.datetime(2024, 4, 16, 1, 13, 29, 857808), datetime_complete=datetime.datetime(2024, 4, 16, 1, 13, 30, 197741), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.216590534664922


In [36]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [37]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.718
train_ibs:  0.217


#### Test

In [38]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [39]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.585


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.221


In [40]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [41]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 01:13:30,611] A new study created in memory with name: no-name-df3e20dc-d8bf-4d10-8dcc-7cbed0b008e1


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.670995670995671
Fold 2 C-index: 0.7053571428571429
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.7637130801687764


[I 2024-04-16 01:13:31,792] A new study created in memory with name: no-name-2982a3b8-5f3e-4008-a200-fc1d6b3c0f80


Fold 5 C-index: 0.7370892018779343
[I 2024-04-16 01:13:31,779] Trial 0 finished with value: 0.7362153329053951 and parameters: {}. Best is trial 0 with value: 0.7362153329053951.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7362153329053951], datetime_start=datetime.datetime(2024, 4, 16, 1, 13, 30, 664215), datetime_complete=datetime.datetime(2024, 4, 16, 1, 13, 31, 776222), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7362153329053951


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.19259004244708322
Fold 2 IBS: 0.15095293415067815
Fold 3 IBS: 0.16506977710039145
Fold 4 IBS: 0.14090021777600747
Fold 5 IBS: 0.15728492161183458
[I 2024-04-16 01:13:33,253] Trial 0 finished with value: 0.161359578617199 and parameters: {}. Best is trial 0 with value: 0.161359578617199.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.161359578617199], datetime_start=datetime.datetime(2024, 4, 16, 1, 13, 31, 832918), datetime_complete=datetime.datetime(2024, 4, 16, 1, 13, 33, 253236), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.161359578617199


In [42]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [43]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.736
train_ibs:  0.161


#### Test

In [44]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [45]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.556


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.251


In [46]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [47]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 01:13:34,222] A new study created in memory with name: no-name-68f3e1e5-e531-448e-a12a-5a5814549f4e


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.670995670995671
Fold 2 C-index: 0.7053571428571429
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.7637130801687764
Fold 5 C-index: 0.7370892018779343
[I 2024-04-16 01:13:35,832] Trial 0 finished with value: 0.7371957250622578 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.7371957250622578.
Fold 1 C-index: 0.6753246753246753
Fold 2 C-index: 0.7008928571428571
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.7679324894514767
Fold 5 C-index: 0.7370892018779343
[I 2024-04-16 01:13:37,580] Trial 1 finished with value: 0.7370321584848789 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.7371957250622578.
Fold 1 C-index: 0.670995670995671
Fold 2 C-index: 0.7008928571428571
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.7679324894514767
Fold 5 C-index: 0.7370892018779343
[I 2024-04-16 01:13:39,721] Trial 2 finished with value: 0.736166357619078 and parameters: {'l1_ratio': 0.22692876841884668}. Best

Fold 2 C-index: 0.7053571428571429
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.7637130801687764
Fold 5 C-index: 0.7370892018779343
[I 2024-04-16 01:14:13,310] Trial 24 finished with value: 0.7371957250622578 and parameters: {'l1_ratio': 0.7602371709740536}. Best is trial 0 with value: 0.7371957250622578.
Fold 1 C-index: 0.6666666666666666
Fold 2 C-index: 0.7053571428571429
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.7637130801687764
Fold 5 C-index: 0.7370892018779343
[I 2024-04-16 01:14:14,327] Trial 25 finished with value: 0.7363299241964569 and parameters: {'l1_ratio': 0.891573855041803}. Best is trial 0 with value: 0.7371957250622578.
Fold 1 C-index: 0.670995670995671
Fold 2 C-index: 0.7053571428571429
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.7637130801687764
Fold 5 C-index: 0.7370892018779343
[I 2024-04-16 01:14:15,390] Trial 26 finished with value: 0.7362153329053951 and parameters: {'l1_ratio': 0.4514399534035586}. Best is trial 0 with value: 0.737195

Fold 1 C-index: 0.670995670995671
Fold 2 C-index: 0.7053571428571429
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.7637130801687764
Fold 5 C-index: 0.7370892018779343
[I 2024-04-16 01:14:45,021] Trial 48 finished with value: 0.7371957250622578 and parameters: {'l1_ratio': 0.5683872043393509}. Best is trial 0 with value: 0.7371957250622578.
Fold 1 C-index: 0.6666666666666666
Fold 2 C-index: 0.7053571428571429
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.7637130801687764
Fold 5 C-index: 0.7370892018779343
[I 2024-04-16 01:14:46,669] Trial 49 finished with value: 0.7363299241964569 and parameters: {'l1_ratio': 0.9505380767046984}. Best is trial 0 with value: 0.7371957250622578.
Fold 1 C-index: 0.670995670995671
Fold 2 C-index: 0.7053571428571429
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.7637130801687764
Fold 5 C-index: 0.7370892018779343
[I 2024-04-16 01:14:47,934] Trial 50 finished with value: 0.7371957250622578 and parameters: {'l1_ratio': 0.7193579141150094}. 

...

Fold 1 IBS: 0.1924091236994006
Fold 2 IBS: 0.15100907458197557
Fold 3 IBS: 0.16515213945655605
Fold 4 IBS: 0.14095407602839238
Fold 5 IBS: 0.15716501008569841
[I 2024-04-16 01:16:15,306] Trial 0 finished with value: 0.16133788477040462 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.16133788477040462.
Fold 1 IBS: 0.19224040788134886
Fold 2 IBS: 0.1513100086548913
Fold 3 IBS: 0.16513327447848067
Fold 4 IBS: 0.14108395736695672
Fold 5 IBS: 0.1569966672751516
[I 2024-04-16 01:16:17,475] Trial 1 finished with value: 0.16135286313136585 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.16133788477040462.
Fold 1 IBS: 0.19225265245976847
Fold 2 IBS: 0.15136771001345142
Fold 3 IBS: 0.16511837524137413
Fold 4 IBS: 0.14110709736340937
Fold 5 IBS: 0.1569595401131213
[I 2024-04-16 01:16:20,399] Trial 2 finished with value: 0.16136107503822494 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 0 with value: 0.1613378847704

Fold 1 IBS: 0.19224653568521086
Fold 2 IBS: 0.1514012960785
Fold 3 IBS: 0.16511102888971094
Fold 4 IBS: 0.14115758164128042
Fold 5 IBS: 0.15692934744828077
[I 2024-04-16 01:17:13,698] Trial 25 finished with value: 0.1613691579485966 and parameters: {'l1_ratio': 0.19986779381542577}. Best is trial 23 with value: 0.1613210805748013.
Fold 1 IBS: 0.19216832011525717
Fold 2 IBS: 0.15152482381793653
Fold 3 IBS: 0.16493778163834757
Fold 4 IBS: 0.14122630371616415
Fold 5 IBS: 0.15682357675923148
[I 2024-04-16 01:17:15,373] Trial 26 finished with value: 0.1613361612093874 and parameters: {'l1_ratio': 0.09330393750339382}. Best is trial 23 with value: 0.1613210805748013.
Fold 1 IBS: 0.1923201307567279
Fold 2 IBS: 0.15123003880949595
Fold 3 IBS: 0.1651550503840734
Fold 4 IBS: 0.14105230435301389
Fold 5 IBS: 0.15704828050001218
[I 2024-04-16 01:17:16,940] Trial 27 finished with value: 0.16136116096066466 and parameters: {'l1_ratio': 0.36900835392127096}. Best is trial 23 with value: 0.161321080574

Fold 5 IBS: 0.15701796152708983
[I 2024-04-16 01:18:18,186] Trial 49 finished with value: 0.16135593936090709 and parameters: {'l1_ratio': 0.355745098499332}. Best is trial 31 with value: 0.16131222622905456.
Fold 1 IBS: 0.19213004749485835
Fold 2 IBS: 0.1516050789806051
Fold 3 IBS: 0.16491435357809608
Fold 4 IBS: 0.14127121223801825
Fold 5 IBS: 0.15677031539860345
[I 2024-04-16 01:18:20,771] Trial 50 finished with value: 0.1613382015380362 and parameters: {'l1_ratio': 0.031172779494489454}. Best is trial 31 with value: 0.16131222622905456.
Fold 1 IBS: 0.192147532060445
Fold 2 IBS: 0.1515959806078035
Fold 3 IBS: 0.1649120166996046
Fold 4 IBS: 0.14126663770372325
Fold 5 IBS: 0.15676415266897656
[I 2024-04-16 01:18:24,929] Trial 51 finished with value: 0.1613372639481106 and parameters: {'l1_ratio': 0.04941055987304063}. Best is trial 31 with value: 0.16131222622905456.
Fold 1 IBS: 0.19215096099775522
Fold 2 IBS: 0.15152443878761593
Fold 3 IBS: 0.16499817043361403
Fold 4 IBS: 0.141229458

Fold 1 IBS: 0.2137111823714941
Fold 2 IBS: 0.22105846696137302
Fold 3 IBS: 0.20417767609022433
Fold 4 IBS: 0.2243242021195586
Fold 5 IBS: 0.21769990553154817
[I 2024-04-16 01:19:30,758] Trial 74 finished with value: 0.21619428661483964 and parameters: {'l1_ratio': 0.002862886448580952}. Best is trial 59 with value: 0.16130915204371993.
Fold 1 IBS: 0.1921422768242721
Fold 2 IBS: 0.15153043397306656
Fold 3 IBS: 0.16499163725478483
Fold 4 IBS: 0.14123447588555174
Fold 5 IBS: 0.15681748493234385
[I 2024-04-16 01:19:33,633] Trial 75 finished with value: 0.1613432617740038 and parameters: {'l1_ratio': 0.10049559697143443}. Best is trial 59 with value: 0.16130915204371993.
Fold 1 IBS: 0.19220519503408265
Fold 2 IBS: 0.1513921270806002
Fold 3 IBS: 0.1651076780140971
Fold 4 IBS: 0.14114448362721096
Fold 5 IBS: 0.1569643027170594
[I 2024-04-16 01:19:36,018] Trial 76 finished with value: 0.16136275729461008 and parameters: {'l1_ratio': 0.18858975508554424}. Best is trial 59 with value: 0.16130915

Fold 5 IBS: 0.15675610636337534
[I 2024-04-16 01:20:44,785] Trial 98 finished with value: 0.16130788174294142 and parameters: {'l1_ratio': 0.07833994485626661}. Best is trial 98 with value: 0.16130788174294142.
Fold 1 IBS: 0.1921899313200978
Fold 2 IBS: 0.15142680765304933
Fold 3 IBS: 0.1650794369327764
Fold 4 IBS: 0.14116786452124977
Fold 5 IBS: 0.15692753735916443
[I 2024-04-16 01:20:48,375] Trial 99 finished with value: 0.16135831555726754 and parameters: {'l1_ratio': 0.16479853152459903}. Best is trial 98 with value: 0.16130788174294142.


* Best trial for IBS: 
 FrozenTrial(number=98, state=TrialState.COMPLETE, values=[0.16130788174294142], datetime_start=datetime.datetime(2024, 4, 16, 1, 20, 41, 938570), datetime_complete=datetime.datetime(2024, 4, 16, 1, 20, 44, 784242), params={'l1_ratio': 0.07833994485626661}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'l1_ratio': FloatDistribution(high=1.0, log=False, low=0.0001, step=None)}, trial_id=98, value=Non

In [48]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [49]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.737
train_ibs:  0.161


#### Test

In [50]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [51]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.6964995386793018)

test_cindex : 0.558


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.07833994485626661)

test_ibs:  0.251


In [52]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [53]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 01:20:50,539] A new study created in memory with name: no-name-e98a47c0-a5ae-48bd-803c-f104343ea609


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7619047619047619
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.7401960784313726
Fold 4 C-index: 0.6814345991561181
Fold 5 C-index: 0.784037558685446
[I 2024-04-16 01:21:19,744] Trial 0 finished with value: 0.7461931710641112 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.7461931710641112.
Fold 1 C-index: 0.7402597402597403
Fold 2 C-index: 0.7098214285714286
Fold 3 C-index: 0.7794117647058824
Fold 4 C-index: 0.70042194092827
Fold 5 C-index: 0.8028169014084507
[I 2024-04-16 01:21:37,302] Trial 1 finished with value: 0.7465463551747543 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 'ma

Fold 1 C-index: 0.7186147186147186
Fold 2 C-index: 0.8125
Fold 3 C-index: 0.8333333333333334
Fold 4 C-index: 0.8523206751054853
Fold 5 C-index: 0.8779342723004695
[I 2024-04-16 01:25:03,484] Trial 16 finished with value: 0.8189405998708015 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 7, 'min_samples_leaf': 1, 'max_depth': 7, 'n_estimators': 498, 'oob_score': True, 'max_samples': 0.36871324569404207, 'max_features': 'log2', 'min_weight_fraction_leaf': 8.824967518839032e-05, 'warm_start': True}. Best is trial 16 with value: 0.8189405998708015.
Fold 1 C-index: 0.7229437229437229
Fold 2 C-index: 0.8080357142857143
Fold 3 C-index: 0.7696078431372549
Fold 4 C-index: 0.7257383966244726
Fold 5 C-index: 0.8215962441314554
[I 2024-04-16 01:25:19,229] Trial 17 finished with value: 0.769584384224524 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 15, 'min_samples_leaf': 1, 'max_depth': 1, 'n_estimators': 493, 'oob_score': True, 'max_samples': 0.27271989967973836, 'max_feat

Fold 1 C-index: 0.7316017316017316
Fold 2 C-index: 0.8169642857142857
Fold 3 C-index: 0.8382352941176471
Fold 4 C-index: 0.8185654008438819
Fold 5 C-index: 0.863849765258216
[I 2024-04-16 01:27:02,733] Trial 31 finished with value: 0.8138432955071524 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 7, 'min_samples_leaf': 4, 'max_depth': 9, 'n_estimators': 457, 'oob_score': True, 'max_samples': 0.4110140220080125, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.05516005867190663, 'warm_start': True}. Best is trial 16 with value: 0.8189405998708015.
Fold 1 C-index: 0.7619047619047619
Fold 2 C-index: 0.8125
Fold 3 C-index: 0.7990196078431373
Fold 4 C-index: 0.7679324894514767
Fold 5 C-index: 0.8450704225352113
[I 2024-04-16 01:27:09,756] Trial 32 finished with value: 0.7972854563469174 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 7, 'min_samples_leaf': 4, 'max_depth': 9, 'n_estimators': 426, 'oob_score': True, 'max_samples': 0.5288432021501348, 'max_features'

...

Fold 5 IBS: 0.174421852678517
[I 2024-04-16 01:39:48,979] Trial 31 finished with value: 0.17683392339955897 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 13, 'min_samples_leaf': 14, 'max_depth': 15, 'n_estimators': 148, 'oob_score': False, 'max_samples': 0.7983356867961283, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.1847177555601581}. Best is trial 11 with value: 0.17336119820893553.
Fold 1 IBS: 0.17600208099339373
Fold 2 IBS: 0.17729776794042784
Fold 3 IBS: 0.17705951526759542
Fold 4 IBS: 0.19409996550095074
Fold 5 IBS: 0.17996400193707593
[I 2024-04-16 01:39:55,781] Trial 32 finished with value: 0.18088466632788874 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 16, 'min_samples_leaf': 18, 'max_depth': 15, 'n_estimators': 129, 'oob_score': True, 'max_samples': 0.7398519463537357, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.14512201637701944}. Best is trial 11 with value: 0.17336119820893553.
Fold 1 IBS: 0.18069568024347987
Fold 2 IBS: 0.

Fold 1 IBS: 0.17444209352625056
Fold 2 IBS: 0.16213459687962897
Fold 3 IBS: 0.18658029015341887
Fold 4 IBS: 0.19096052126724972
Fold 5 IBS: 0.160743603219205
[I 2024-04-16 01:42:08,122] Trial 47 finished with value: 0.1749722210091506 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 19, 'min_samples_leaf': 6, 'max_depth': 17, 'n_estimators': 48, 'oob_score': False, 'max_samples': 0.5836246358507913, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.08838589682559546}. Best is trial 11 with value: 0.17336119820893553.
Fold 1 IBS: 0.21394896398369084
Fold 2 IBS: 0.22096284686613055
Fold 3 IBS: 0.20483718884499716
Fold 4 IBS: 0.2248454593696118
Fold 5 IBS: 0.21856174512063364
[I 2024-04-16 01:42:13,582] Trial 48 finished with value: 0.2166312408370128 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 12, 'max_depth': 19, 'n_estimators': 124, 'oob_score': False, 'max_samples': 0.3075018204661719, 'max_features': 'log2', 'min_weight_fraction_l

Fold 5 IBS: 0.1716386754696866
[I 2024-04-16 01:49:23,658] Trial 62 finished with value: 0.17621006766938516 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 19, 'n_estimators': 421, 'oob_score': True, 'max_samples': 0.8259912471429802, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.2092308756694655}. Best is trial 11 with value: 0.17336119820893553.
Fold 1 IBS: 0.17415213320463757
Fold 2 IBS: 0.1617279283932607
Fold 3 IBS: 0.18375007806639526
Fold 4 IBS: 0.18093667575423525
Fold 5 IBS: 0.16750194117016448
[I 2024-04-16 01:49:43,838] Trial 63 finished with value: 0.17361375131773865 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 19, 'min_samples_leaf': 1, 'max_depth': 20, 'n_estimators': 474, 'oob_score': True, 'max_samples': 0.7851415216753531, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.16951944917135664}. Best is trial 11 with value: 0.17336119820893553.
Fold 1 IBS: 0.18036142871724786
Fold 2 IBS: 0.16206

Fold 1 IBS: 0.17324342127994388
Fold 2 IBS: 0.16145074542296425
Fold 3 IBS: 0.1833602919336159
Fold 4 IBS: 0.1795050214021496
Fold 5 IBS: 0.16745400415861386
[I 2024-04-16 01:53:35,377] Trial 78 finished with value: 0.17300269683945751 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 17, 'min_samples_leaf': 6, 'max_depth': 16, 'n_estimators': 406, 'oob_score': True, 'max_samples': 0.5519676952991933, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.11560830509081324}. Best is trial 78 with value: 0.17300269683945751.
Fold 1 IBS: 0.18308493469262166
Fold 2 IBS: 0.17752781133054824
Fold 3 IBS: 0.17865493574690022
Fold 4 IBS: 0.1934599594730117
Fold 5 IBS: 0.1827420656656263
[I 2024-04-16 01:53:48,824] Trial 79 finished with value: 0.18309394138174165 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 16, 'min_samples_leaf': 6, 'max_depth': 16, 'n_estimators': 402, 'oob_score': True, 'max_samples': 0.5136803098501763, 'max_features': 'log2', 'min_weight_fraction_l

Fold 5 IBS: 0.16853370354637134
[I 2024-04-16 01:56:47,663] Trial 93 finished with value: 0.17452675084449457 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 16, 'min_samples_leaf': 8, 'max_depth': 9, 'n_estimators': 382, 'oob_score': True, 'max_samples': 0.39259915650398747, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.0007655640662008745}. Best is trial 84 with value: 0.17234548100766184.
Fold 1 IBS: 0.17728796339095554
Fold 2 IBS: 0.1618275633358139
Fold 3 IBS: 0.18252336449392226
Fold 4 IBS: 0.17492094283521942
Fold 5 IBS: 0.1646569791557643
[I 2024-04-16 01:56:56,831] Trial 94 finished with value: 0.17224336264233508 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 15, 'min_samples_leaf': 7, 'max_depth': 11, 'n_estimators': 397, 'oob_score': True, 'max_samples': 0.401310433648782, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.017290020666363863}. Best is trial 94 with value: 0.17224336264233508.
Fold 1 IBS: 0.17621008692601267
Fold 2 IBS: 0.

In [54]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [55]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.849
train_ibs:  0.172


#### Test

In [56]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [57]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=15, max_features=None, max_leaf_nodes=17,
                     max_samples=0.9830923957685131, min_samples_leaf=4,
                     min_samples_split=11,
                     min_weight_fraction_leaf=0.04949739725380848,
                     n_estimators=283, random_state=123, warm_start=True)

test_cindex:  0.555


RandomSurvivalForest(max_depth=11, max_features='log2', max_leaf_nodes=15,
                     max_samples=0.401310433648782, min_samples_leaf=7,
                     min_samples_split=13,
                     min_weight_fraction_leaf=0.017290020666363863,
                     n_estimators=397, oob_score=True, random_state=123)

test_ibs:  0.226


In [58]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [59]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [60]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 01:57:58,647] A new study created in memory with name: no-name-0da87e23-a934-499f-9984-0cf852fcf2bd


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.658008658008658
Fold 2 C-index: 0.7723214285714286
Fold 3 C-index: 0.8137254901960784
Fold 4 C-index: 0.7257383966244726
Fold 5 C-index: 0.8028169014084507
[I 2024-04-16 01:58:00,918] Trial 0 finished with value: 0.7545221749618177 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7545221749618177.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 01:58:05,871] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531}

Fold 5 C-index: 0.7511737089201878
[I 2024-04-16 01:58:57,552] Trial 15 finished with value: 0.7300192908165389 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 9, 'min_samples_leaf': 13, 'max_depth': 4, 'n_estimators': 258, 'oob_score': False, 'warm_start': True, 'max_features': 'sqrt', 'max_samples': 0.6880212408103346, 'min_weight_fraction_leaf': 0.07884420972256234}. Best is trial 12 with value: 0.7836335847164032.
Fold 1 C-index: 0.6385281385281385
Fold 2 C-index: 0.7276785714285714
Fold 3 C-index: 0.7549019607843137
Fold 4 C-index: 0.7088607594936709
Fold 5 C-index: 0.7370892018779343
[I 2024-04-16 01:59:00,087] Trial 16 finished with value: 0.7134117264225257 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.8750346354626457, 'min_weight_fraction_leaf': 0.4093818399278544}. Best is trial 12 with value: 0.7836335847

Fold 1 C-index: 0.645021645021645
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.75
Fold 4 C-index: 0.7088607594936709
Fold 5 C-index: 0.6807511737089202
[I 2024-04-16 01:59:41,812] Trial 30 finished with value: 0.7060338585019901 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 3, 'min_samples_leaf': 3, 'max_depth': 5, 'n_estimators': 355, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.23596047192138714, 'min_weight_fraction_leaf': 0.09139810961881356}. Best is trial 12 with value: 0.7836335847164032.
Fold 1 C-index: 0.6623376623376623
Fold 2 C-index: 0.7544642857142857
Fold 3 C-index: 0.8088235294117647
Fold 4 C-index: 0.7383966244725738
Fold 5 C-index: 0.812206572769953
[I 2024-04-16 01:59:44,030] Trial 31 finished with value: 0.755245734941248 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 11, 'min_samples_leaf': 6, 'max_depth': 13, 'n_estimators': 453, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'ma

...

Fold 1 IBS: 0.19574948384138363
Fold 2 IBS: 0.18320445235447416
Fold 3 IBS: 0.18399571264411738
Fold 4 IBS: 0.18818584834971439
Fold 5 IBS: 0.1847924741624137
[I 2024-04-16 02:07:02,658] Trial 30 finished with value: 0.18718559427042067 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 3, 'min_samples_leaf': 4, 'max_depth': 12, 'n_estimators': 355, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7262614295600427, 'min_weight_fraction_leaf': 0.020774312173092817}. Best is trial 23 with value: 0.17858540517959293.
Fold 1 IBS: 0.2012130462314688
Fold 2 IBS: 0.19251954390900003
Fold 3 IBS: 0.1894012799573833
Fold 4 IBS: 0.199635572696493
Fold 5 IBS: 0.19581363629423457
[I 2024-04-16 02:07:09,738] Trial 31 finished with value: 0.19571661581771593 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 2, 'min_samples_leaf': 4, 'max_depth': 12, 'n_estimators': 396, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.72

Fold 1 IBS: 0.18805213459148196
Fold 2 IBS: 0.1681544887707249
Fold 3 IBS: 0.18188248727268916
Fold 4 IBS: 0.16335868140621157
Fold 5 IBS: 0.16065070124373665
[I 2024-04-16 02:09:36,918] Trial 45 finished with value: 0.17241969865696888 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 9, 'min_samples_leaf': 2, 'max_depth': 8, 'n_estimators': 433, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.9991372338226472, 'min_weight_fraction_leaf': 0.0747513303047341}. Best is trial 45 with value: 0.17241969865696888.
Fold 1 IBS: 0.20355292029098285
Fold 2 IBS: 0.204107363403046
Fold 3 IBS: 0.19458831147904218
Fold 4 IBS: 0.2080676178213458
Fold 5 IBS: 0.20054223894272527
[I 2024-04-16 02:09:46,443] Trial 46 finished with value: 0.20217169038742844 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 10, 'min_samples_leaf': 3, 'max_depth': 5, 'n_estimators': 368, 'oob_score': True, 'warm_start': False, 'max_features': 0.1, 'max_samples': 0.99396622

Fold 1 IBS: 0.1958742848832495
Fold 2 IBS: 0.16658576851987503
Fold 3 IBS: 0.18445996749161686
Fold 4 IBS: 0.1639381964146127
Fold 5 IBS: 0.15665286759755212
[I 2024-04-16 02:10:51,426] Trial 60 finished with value: 0.17350221698138124 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 9, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 144, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.8720069627631468, 'min_weight_fraction_leaf': 0.012040362487536843}. Best is trial 51 with value: 0.17164165457052566.
Fold 1 IBS: 0.2032968814717009
Fold 2 IBS: 0.16957221694600605
Fold 3 IBS: 0.18138481693412806
Fold 4 IBS: 0.16894879325917614
Fold 5 IBS: 0.1607009374338522
[I 2024-04-16 02:10:52,128] Trial 61 finished with value: 0.17678072920897267 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 8, 'min_samples_leaf': 1, 'max_depth': 8, 'n_estimators': 12, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.9514836

Fold 1 IBS: 0.18861014255452424
Fold 2 IBS: 0.1845345537529862
Fold 3 IBS: 0.18346887733887327
Fold 4 IBS: 0.18717174704010517
Fold 5 IBS: 0.18266022048955657
[I 2024-04-16 02:12:52,733] Trial 75 finished with value: 0.1852891082352091 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 6, 'min_samples_leaf': 2, 'max_depth': 10, 'n_estimators': 422, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.9911930954780681, 'min_weight_fraction_leaf': 0.09946408845915759}. Best is trial 51 with value: 0.17164165457052566.
Fold 1 IBS: 0.19999952895742903
Fold 2 IBS: 0.19525046896204418
Fold 3 IBS: 0.18771598274655127
Fold 4 IBS: 0.20216609938880742
Fold 5 IBS: 0.1939284836539042
[I 2024-04-16 02:13:01,158] Trial 76 finished with value: 0.19581211274174723 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 7, 'min_samples_leaf': 8, 'max_depth': 12, 'n_estimators': 391, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.3

Fold 1 IBS: 0.19091350642673613
Fold 2 IBS: 0.17241215632395923
Fold 3 IBS: 0.1790391638097396
Fold 4 IBS: 0.17759959680796716
Fold 5 IBS: 0.172995935264056
[I 2024-04-16 02:14:30,434] Trial 90 finished with value: 0.17859207172649164 and parameters: {'min_samples_split': 11, 'max_leaf_nodes': 5, 'min_samples_leaf': 2, 'max_depth': 3, 'n_estimators': 257, 'oob_score': True, 'warm_start': False, 'max_features': 'auto', 'max_samples': 0.7723359182067533, 'min_weight_fraction_leaf': 0.03269650735672842}. Best is trial 51 with value: 0.17164165457052566.
Fold 1 IBS: 0.19076745166131998
Fold 2 IBS: 0.16416395573150727
Fold 3 IBS: 0.18090486926599017
Fold 4 IBS: 0.16213093075559837
Fold 5 IBS: 0.16216887109996253
[I 2024-04-16 02:14:37,957] Trial 91 finished with value: 0.17202721570287566 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 6, 'min_samples_leaf': 1, 'max_depth': 7, 'n_estimators': 264, 'oob_score': True, 'warm_start': False, 'max_features': None, 'max_samples': 0.909

In [61]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [62]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.839
train_ibs:  0.172


#### Test

In [63]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [64]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=7, max_features='auto', max_leaf_nodes=14,
                   max_samples=0.9766144605477189, min_samples_leaf=2,
                   min_samples_split=2,
                   min_weight_fraction_leaf=0.030383657143011084,
                   n_estimators=80, oob_score=True, random_state=123,
                   warm_start=True)

C-index score: 0.574


ExtraSurvivalTrees(max_depth=11, max_features=None, max_leaf_nodes=7,
                   max_samples=0.8949711629363306, min_samples_leaf=2,
                   min_samples_split=10,
                   min_weight_fraction_leaf=0.08881715885165918,
                   n_estimators=120, oob_score=True, random_state=123)

IBS: 0.222


In [65]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [66]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-16 02:15:30,266] A new study created in memory with name: no-name-f6d81090-3117-4321-b6ab-c182a12550de


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 02:16:08,012] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 02:16:27,618] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 02:26:20,044] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 12 with value: 0.7476370623899082.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 02:27:33,901] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'squar

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 02:40:43,238] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.8840318412875596, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.27382248438555523, 'n_estimators': 385, 'criterion': 'squared_error', 'ccp_alpha': 2.0183033060186855, 'min_weight_fraction_leaf': 0.33643534713187806, 'max_features': 'auto', 'min_impurity_decrease': 5.889654690360788e-06, 'validation_fraction': 0.8062622793646869, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 12 with value: 0.7476370623899082.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-16 02:41:52,347] Trial 26 finished with value: 0.5 and parameters: {'subsample': 0.7565765917190008, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.4300954216773497, 'n_estimators': 440, 'criterion': 'friedma

...

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-16 04:51:18,473] Trial 55 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.809323041898869, 'learning_rate': 0.0040826970179266685, 'dropout_rate': 0.20419816136861105, 'n_estimators': 437, 'criterion': 'squared_error', 'ccp_alpha': 1.5125790196698774, 'min_weight_fraction_leaf': 0.21953014805517473, 'max_features': 'auto', 'min_impurity_decrease': 2.0868015436948727e-05, 'validation_fraction': 0.9659902853562541, 'min_samples_split': 19, 'max_leaf_nodes': 18, 'min_samples_leaf': 8, 'max_depth': 2}. Best is trial 9 with value: 0.21540040804416147.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667935
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609582
[I 2024-04-16 04:51:57,110] Trial 56 finished with value: 0.21659054862241592 and parameters: {'subsample': 0.2887765610954624, 'learning_rate': 0.09850922

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.2181243152560957
[I 2024-04-16 05:00:14,711] Trial 66 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.5392724474754766, 'learning_rate': 0.01461816521127875, 'dropout_rate': 0.22076051954616932, 'n_estimators': 377, 'criterion': 'squared_error', 'ccp_alpha': 0.8099144041738244, 'min_weight_fraction_leaf': 0.03518122515344503, 'max_features': 'sqrt', 'min_impurity_decrease': 1.1843050264198749e-07, 'validation_fraction': 0.9678192460592685, 'min_samples_split': 20, 'max_leaf_nodes': 16, 'min_samples_leaf': 11, 'max_depth': 2}. Best is trial 63 with value: 0.21422377695518507.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.2181243152560957
[I 2024-04-16 05:01:01,557] Trial 67 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.680855021230625, 'learning_rate': 0.0049198830

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-16 05:04:50,196] Trial 77 finished with value: 0.21659054862241583 and parameters: {'subsample': 0.5357913465602108, 'learning_rate': 0.052009701795490706, 'dropout_rate': 0.311272991332878, 'n_estimators': 175, 'criterion': 'squared_error', 'ccp_alpha': 0.9488717637120231, 'min_weight_fraction_leaf': 0.016779792096412772, 'max_features': 'sqrt', 'min_impurity_decrease': 3.886076395376554e-06, 'validation_fraction': 0.9332646380443554, 'min_samples_split': 4, 'max_leaf_nodes': 11, 'min_samples_leaf': 5, 'max_depth': 17}. Best is trial 75 with value: 0.2053055413081169.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.2181243152560957
[I 2024-04-16 05:05:05,525] Trial 78 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.692374586728691, 'learning_rate': 0.0423105906235

Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.2181243152560957
[I 2024-04-16 05:07:17,064] Trial 88 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.5987701753734233, 'learning_rate': 0.050213780826442185, 'dropout_rate': 0.1938411005791284, 'n_estimators': 139, 'criterion': 'squared_error', 'ccp_alpha': 0.3180081639828751, 'min_weight_fraction_leaf': 0.01874802778250728, 'max_features': None, 'min_impurity_decrease': 1.0777511909570898e-07, 'validation_fraction': 0.9901236820423219, 'min_samples_split': 5, 'max_leaf_nodes': 10, 'min_samples_leaf': 9, 'max_depth': 13}. Best is trial 85 with value: 0.20271375419835502.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.2181243152560957
[I 2024-04-16 05:07:26,747] Trial 89 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.6527388011508777, 'learning_rate': 0.04043979612509

Fold 3 IBS: 0.19654120805549916
Fold 4 IBS: 0.20962124167390286
Fold 5 IBS: 0.20124873409085406
[I 2024-04-16 05:09:23,792] Trial 99 finished with value: 0.201975871333619 and parameters: {'subsample': 0.7090975652007764, 'learning_rate': 0.05019766261164787, 'dropout_rate': 0.10216550214025771, 'n_estimators': 257, 'criterion': 'squared_error', 'ccp_alpha': 0.0045100477558669, 'min_weight_fraction_leaf': 0.09258479081501242, 'max_features': 'sqrt', 'min_impurity_decrease': 8.264881459144936e-07, 'validation_fraction': 0.6748895535913024, 'min_samples_split': 3, 'max_leaf_nodes': 17, 'min_samples_leaf': 6, 'max_depth': 15}. Best is trial 99 with value: 0.201975871333619.


* Best trial for IBS: 
 FrozenTrial(number=99, state=TrialState.COMPLETE, values=[0.201975871333619], datetime_start=datetime.datetime(2024, 4, 16, 5, 9, 6, 115873), datetime_complete=datetime.datetime(2024, 4, 16, 5, 9, 23, 791252), params={'subsample': 0.7090975652007764, 'learning_rate': 0.05019766261164787, 'drop

In [67]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [68]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.751
train_ibs:  0.202


#### Test

In [69]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [70]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.0188293664905176,
                                 criterion='squared_error',
                                 dropout_rate=0.10424287148225743,
                                 learning_rate=0.03173728343640762, max_depth=9,
                                 max_features='auto', max_leaf_nodes=17,
                                 min_impurity_decrease=1.3598537758171917e-07,
                                 min_samples_leaf=11, min_samples_split=17,
                                 min_weight_fraction_leaf=0.3120175734494814,
                                 n_estimators=469, random_state=123,
                                 subsample=0.7639525884887477,
                                 validation_fraction=0.652629801149938)

C-index score: 0.576


GradientBoostingSurvivalAnalysis(ccp_alpha=0.0045100477558669,
                                 criterion='squared_error',
                                 dropout_rate=0.10216550214025771,
                                 learning_rate=0.05019766261164787,
                                 max_depth=15, max_features='sqrt',
                                 max_leaf_nodes=17,
                                 min_impurity_decrease=8.264881459144936e-07,
                                 min_samples_leaf=6, min_samples_split=3,
                                 min_weight_fraction_leaf=0.09258479081501242,
                                 n_estimators=257, random_state=123,
                                 subsample=0.7090975652007764,
                                 validation_fraction=0.6748895535913024)

IBS: 0.217


In [71]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [72]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [73]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = StandardScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-16 05:09:37,868] A new study created in memory with name: no-name-2cc4e40e-032f-43d3-afd7-cd2c5fbfb160


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7549019607843137
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.7323943661971831
[I 2024-04-16 05:09:38,516] Trial 0 finished with value: 0.6925028294272052 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6925028294272052.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.75
Fold 3 C-index: 0.75
Fold 4 C-index: 0.6624472573839663
Fold 5 C-index: 0.7323943661971831
[I 2024-04-16 05:09:43,889] Trial 1 finished with value: 0.6915224372703425 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.6925028294272052.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7549019607843137
Fold 4 C-index: 0.6708860759493671
Fold 5 C-index: 0.73708920

Fold 1 C-index: 0.6017316017316018
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.7598039215686274
Fold 4 C-index: 0.679324894514768
Fold 5 C-index: 0.7511737089201878
[I 2024-04-16 05:10:34,278] Trial 19 finished with value: 0.7066211110613226 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.34721836740782674, 'n_estimators': 401, 'learning_rate': 0.08684988472507524}. Best is trial 16 with value: 0.7074160176315457.
Fold 1 C-index: 0.5584415584415584
Fold 2 C-index: 0.75
Fold 3 C-index: 0.7549019607843137
Fold 4 C-index: 0.6708860759493671
Fold 5 C-index: 0.7511737089201878
[I 2024-04-16 05:10:38,341] Trial 20 finished with value: 0.6970806608190854 and parameters: {'subsample': 0.30794140712521173, 'dropout_rate': 0.5777838682216713, 'n_estimators': 420, 'learning_rate': 0.06551866052750378}. Best is trial 16 with value: 0.7074160176315457.
Fold 1 C-index: 0.6017316017316018
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.7647058823529411
Fold 4 C-inde

Fold 1 C-index: 0.5584415584415584
Fold 2 C-index: 0.7455357142857143
Fold 3 C-index: 0.7549019607843137
Fold 4 C-index: 0.6708860759493671
Fold 5 C-index: 0.7464788732394366
[I 2024-04-16 05:11:42,886] Trial 38 finished with value: 0.695248836540078 and parameters: {'subsample': 0.2711106835858654, 'dropout_rate': 0.6515731633509281, 'n_estimators': 342, 'learning_rate': 0.06766859442184275}. Best is trial 24 with value: 0.7083964097884085.
Fold 1 C-index: 0.5757575757575758
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.7598039215686274
Fold 4 C-index: 0.6877637130801688
Fold 5 C-index: 0.7417840375586855
[I 2024-04-16 05:11:48,196] Trial 39 finished with value: 0.7012361353072973 and parameters: {'subsample': 0.1899832825436063, 'dropout_rate': 0.7545555141153719, 'n_estimators': 460, 'learning_rate': 0.05598015324329953}. Best is trial 24 with value: 0.7083964097884085.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.7598039215686274
Fo

...

Fold 3 IBS: 0.17559695146919774
Fold 4 IBS: 0.19136347113687585
Fold 5 IBS: 0.18238485830356638
[I 2024-04-16 05:15:21,304] Trial 19 finished with value: 0.18561392095155274 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.9930703343219778, 'n_estimators': 51, 'learning_rate': 0.04096883532946153}. Best is trial 19 with value: 0.18561392095155274.
Fold 1 IBS: 0.20385002842262298
Fold 2 IBS: 0.17898541076661173
Fold 3 IBS: 0.17678497782327718
Fold 4 IBS: 0.1964113675862836
Fold 5 IBS: 0.18635022380429012
[I 2024-04-16 05:15:21,587] Trial 20 finished with value: 0.18847640168061713 and parameters: {'subsample': 0.1435040576337751, 'dropout_rate': 0.7933084651006226, 'n_estimators': 43, 'learning_rate': 0.04127909023805986}. Best is trial 19 with value: 0.18561392095155274.
Fold 1 IBS: 0.20152752504533955
Fold 2 IBS: 0.186025328567609
Fold 3 IBS: 0.18245538571556968
Fold 4 IBS: 0.19968663830097044
Fold 5 IBS: 0.19044129437999457
[I 2024-04-16 05:15:21,827] Trial 21 fin

Fold 4 IBS: 0.19970147648879832
Fold 5 IBS: 0.19379867220842292
[I 2024-04-16 05:15:34,671] Trial 38 finished with value: 0.19427302171643077 and parameters: {'subsample': 0.10045702238474481, 'dropout_rate': 0.2908235532799638, 'n_estimators': 117, 'learning_rate': 0.01019431388894447}. Best is trial 19 with value: 0.18561392095155274.
Fold 1 IBS: 0.23308107527388047
Fold 2 IBS: 0.18915589279847106
Fold 3 IBS: 0.17859698469690719
Fold 4 IBS: 0.21592434119861256
Fold 5 IBS: 0.17571592320871024
[I 2024-04-16 05:15:36,589] Trial 39 finished with value: 0.19849484343531631 and parameters: {'subsample': 0.167666800643059, 'dropout_rate': 0.531096799684823, 'n_estimators': 267, 'learning_rate': 0.019230973079348387}. Best is trial 19 with value: 0.18561392095155274.
Fold 1 IBS: 0.2526994588964834
Fold 2 IBS: 0.23678037479229275
Fold 3 IBS: 0.19647581845865703
Fold 4 IBS: 0.26532918185085924
Fold 5 IBS: 0.19634674564701443
[I 2024-04-16 05:15:37,656] Trial 40 finished with value: 0.229526315

Fold 4 IBS: 0.23524556713861994
Fold 5 IBS: 0.18175978216705466
[I 2024-04-16 05:15:53,932] Trial 57 finished with value: 0.20702822992486106 and parameters: {'subsample': 0.287480927064245, 'dropout_rate': 0.310570838083683, 'n_estimators': 256, 'learning_rate': 0.019746883319291475}. Best is trial 19 with value: 0.18561392095155274.
Fold 1 IBS: 0.20873751439183064
Fold 2 IBS: 0.2098077449702301
Fold 3 IBS: 0.19579887472650773
Fold 4 IBS: 0.21662065248561807
Fold 5 IBS: 0.20963911462011556
[I 2024-04-16 05:15:54,107] Trial 58 finished with value: 0.2081207802388604 and parameters: {'subsample': 0.2341458361742736, 'dropout_rate': 0.6276973088855227, 'n_estimators': 13, 'learning_rate': 0.03237151215124951}. Best is trial 19 with value: 0.18561392095155274.
Fold 1 IBS: 0.20156112265850423
Fold 2 IBS: 0.18998875782566582
Fold 3 IBS: 0.1802717139071199
Fold 4 IBS: 0.20048365943759167
Fold 5 IBS: 0.1891019830466934
[I 2024-04-16 05:15:54,381] Trial 59 finished with value: 0.19228144737511

Fold 1 IBS: 0.2123118865289582
Fold 2 IBS: 0.18711899052237113
Fold 3 IBS: 0.17344631354825793
Fold 4 IBS: 0.20915818709182382
Fold 5 IBS: 0.18581168448565522
[I 2024-04-16 05:16:03,592] Trial 77 finished with value: 0.19356941243541326 and parameters: {'subsample': 0.5189609406746745, 'dropout_rate': 0.4517032081036193, 'n_estimators': 39, 'learning_rate': 0.05699325023464598}. Best is trial 73 with value: 0.1829482691105161.
Fold 1 IBS: 0.21245690064091996
Fold 2 IBS: 0.1716813809222342
Fold 3 IBS: 0.1726090449523327
Fold 4 IBS: 0.1987968002916969
Fold 5 IBS: 0.17435871466828684
[I 2024-04-16 05:16:04,034] Trial 78 finished with value: 0.18598056829509413 and parameters: {'subsample': 0.1289194344931569, 'dropout_rate': 0.3814766477624397, 'n_estimators': 66, 'learning_rate': 0.05240930822810723}. Best is trial 73 with value: 0.1829482691105161.
Fold 1 IBS: 0.21183884213778528
Fold 2 IBS: 0.21744879587985189
Fold 3 IBS: 0.2006226196411385
Fold 4 IBS: 0.22028923854047944
Fold 5 IBS: 0

Fold 2 IBS: 0.17123814838553747
Fold 3 IBS: 0.17269115495453874
Fold 4 IBS: 0.18475235232092624
Fold 5 IBS: 0.17313385137825504
[I 2024-04-16 05:16:21,248] Trial 96 finished with value: 0.18169342721982126 and parameters: {'subsample': 0.10080401063092843, 'dropout_rate': 0.3564078078462598, 'n_estimators': 84, 'learning_rate': 0.039295022656166335}. Best is trial 96 with value: 0.18169342721982126.
Fold 1 IBS: 0.27573287205708735
Fold 2 IBS: 0.2714531332947983
Fold 3 IBS: 0.23591311125478967
Fold 4 IBS: 0.25811998045479884
Fold 5 IBS: 0.23209888832642764
[I 2024-04-16 05:16:24,362] Trial 97 finished with value: 0.25466359707758035 and parameters: {'subsample': 0.22570028642335044, 'dropout_rate': 0.31372183756474226, 'n_estimators': 328, 'learning_rate': 0.04015331233811272}. Best is trial 96 with value: 0.18169342721982126.
Fold 1 IBS: 0.22218380231437124
Fold 2 IBS: 0.18900802159589356
Fold 3 IBS: 0.1705539059758543
Fold 4 IBS: 0.22288816270765116
Fold 5 IBS: 0.18390427105879292
[I 

In [74]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [75]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.713
train_ibs:  0.182


#### Test

In [76]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [77]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.3170885754967726,
                                              learning_rate=0.017271336468030765,
                                              n_estimators=364,
                                              random_state=123,
                                              subsample=0.11004058121439181)

C-index score: 0.576


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.3564078078462598,
                                              learning_rate=0.039295022656166335,
                                              n_estimators=84, random_state=123,
                                              subsample=0.10080401063092843)

IBS: 0.229


In [78]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

# Results

In [79]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
Randomsurvivalforest,0.849,1.0
ExtraSurvivalTrees,0.839,2.0
GradientBoosting,0.751,3.0
CoxElastic,0.737,4.0
CoxLasso,0.736,5.0
CoxPH,0.735,6.0
CoxRidge,0.718,7.0
ComponentwiseGradientBoosting,0.713,8.0


In [80]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
CoxLasso,0.161,1.5
CoxElastic,0.161,1.5
CoxPH,0.162,3.0
Randomsurvivalforest,0.172,4.5
ExtraSurvivalTrees,0.172,4.5
ComponentwiseGradientBoosting,0.182,6.0
GradientBoosting,0.202,7.0
CoxRidge,0.217,8.0


In [81]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
CoxRidge,0.585,1.0
GradientBoosting,0.576,2.5
ComponentwiseGradientBoosting,0.576,2.5
ExtraSurvivalTrees,0.574,4.0
CoxPH,0.558,5.5
CoxElastic,0.558,5.5
CoxLasso,0.556,7.0
Randomsurvivalforest,0.555,8.0


In [82]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
GradientBoosting,0.217,1.0
CoxRidge,0.221,2.0
ExtraSurvivalTrees,0.222,3.0
Randomsurvivalforest,0.226,4.0
ComponentwiseGradientBoosting,0.229,5.0
CoxLasso,0.251,6.5
CoxElastic,0.251,6.5
CoxPH,0.252,8.0


In [85]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = 'path_to_your_folder/'  

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d2/os/standard/plsr/' 

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d2_os_standard_plsr_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [86]:
from datetime import date

current_date = date.today()
print(current_date)

2024-04-16
